# Extended Lab: Multiple Linear Regression for Economic Analysis
## Modeling Restaurant Profitability from City Population and Median Household Income

**Economics framing:** You are an economic analyst for a restaurant chain considering expansion.
Rather than population alone (the original one-variable lab), you now have two economic
indicators for each candidate city:

- $x_1$ = **population** (in units of 10,000s of people)
- $x_2$ = **median household income** (in units of \$1,000s)

and the outcome:

- $y$ = **average monthly restaurant profit** (in units of \$10,000s)

This is a classic **multiple linear regression / hedonic-style econometric model**: you're
estimating how much of the variation in profit ("the dependent variable") is explained by two
independent economic variables, and — just as importantly — you're going to interpret the
estimated coefficients the way an economist would: as **marginal effects, holding the other
variable fixed (ceteris paribus)**.

# Outline
- [ 1 - Packages ](#1)
- [ 2 - Problem Statement (Economics) ](#2)
- [ 3 - Dataset ](#3)
- [ 4 - Refresher: Multiple Linear Regression ](#4)
- [ 5 - Compute Cost ](#5)
    - [ Exercise 1 ](#ex01)
- [ 6 - Compute Gradient ](#6)
    - [ Exercise 2 ](#ex02)
- [ 7 - Feature Scaling ](#7)
    - [ Exercise 3 ](#ex03)
- [ 8 - Run Gradient Descent ](#8)
- [ 9 - Sanity Check: Closed-Form (OLS) Solution ](#9)
- [ 10 - Economic Interpretation of Coefficients ](#10)
- [ 11 - Model Fit Diagnostics (R²) ](#11)
- [ 12 - Predicting for New Cities ](#12)
- [ 13 - Extension Questions ](#13)


_**NOTE:** This notebook is fully self-contained — the dataset is generated in code
(no external files needed), so it will run top-to-bottom on its own. Please don't delete the
`### START CODE HERE ###` / `### END CODE HERE ###` markers even in the skeleton version; this
keeps it compatible with autograder-style workflows if you adapt it for a class._


<a name="1"></a>
## 1 - Packages

- [numpy](https://www.numpy.org) for vectorized linear algebra (matrices, dot products).
- [matplotlib](https://matplotlib.org) for plotting.
- `copy`, `math` are standard library utilities used inside gradient descent.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy
import math
%matplotlib inline
np.set_printoptions(precision=4, suppress=True)


<a name="2"></a>
## 2 - Problem Statement (Economics)

Suppose you are the chief economist for a restaurant franchise. The firm wants a **data-driven
rule** for deciding which cities are attractive for a new outlet. Two well-known drivers of
local consumer demand are:

1. **Market size** — proxied by city population. More people nearby generally means more
   potential customers.
2. **Purchasing power** — proxied by median household income. Higher income typically means
   more discretionary spending on dining out.

Economically, we're hypothesizing a **linear demand-side profit function**:

$$ \text{profit} = w_1 \cdot \text{population} + w_2 \cdot \text{income} + b + \varepsilon $$

where $\varepsilon$ is unexplained noise (local competition, rent, marketing, etc. — factors
we haven't measured). Estimating $w_1, w_2, b$ from data is exactly a **multiple linear
regression / OLS (ordinary least squares) problem**, which we'll solve two ways in this lab:
(1) gradient descent (as in the original one-variable lab), and (2) the closed-form normal
equation, as a cross-check — economists usually see OLS in closed form first, so this lets you
connect the two.


<a name="3"></a>
## 3 - Dataset

Since we don't have a live data feed, we simulate a plausible dataset with `generate_economic_data()`
below. This keeps the lab fully reproducible and portable (no CSV/JSON files to manage), while
still behaving like real cross-sectional economic data: two correlated-but-distinct features,
a linear signal, and realistic Gaussian noise.

Replace this cell with `pd.read_csv(...)` / `np.loadtxt(...)` any time you want to run the same
analysis on **your own** city-level data — nothing else in the notebook needs to change, since
`compute_cost`, `compute_gradient`, and the training loop only care about the shapes of `X` and `y`.


In [ ]:
def generate_economic_data(m=100, seed=1):
    """
    Simulates city-level data for a restaurant-expansion economic analysis.

    Features (columns of X):
      x1 = population of city, in units of 10,000s (6.11 -> 61,100 people)
      x2 = median household income of city, in units of $1,000s (45.0 -> $45,000)
    Target (y):
      y  = average monthly profit of a restaurant in that city, in units of $10,000s

    The data-generating process (unknown to the model, known to us since we simulated it) is:
        profit = 1.15 * population + 0.045 * income - 5.2 + noise
    Economically: +10,000 people is worth about +$11,500/month in profit, and +$1,000 in
    median income is worth about +$450/month in profit, holding the other variable fixed.

    Returns:
        X (ndarray): Shape (m, 2) feature matrix [population, income]
        y (ndarray): Shape (m,)   target profit
    """
    rng = np.random.default_rng(seed)
    population = rng.uniform(2, 25, m)     # 20,000 - 250,000 people
    income     = rng.uniform(20, 120, m)   # $20k - $120k median household income
    true_w = np.array([1.15, 0.045])
    true_b = -5.2
    noise = rng.normal(0, 3.0, m)
    profit = true_w[0]*population + true_w[1]*income + true_b + noise
    X = np.column_stack([population, income])
    y = profit
    return X, y

X_train, y_train = generate_economic_data()
print("Type of X_train:", type(X_train))
print("X_train shape:", X_train.shape)
print("First 5 rows of X_train (population, income):\n", X_train[:5])
print("\nFirst 5 values of y_train (profit):\n", y_train[:5])


#### Check dimensions

Multiple linear regression generalizes the one-variable case: instead of a 1-D array `x_train`
of shape `(m,)`, we now have a 2-D **design matrix** `X_train` of shape `(m, n)`, where `m` is
the number of training examples (cities) and `n` is the number of features (here, `n=2`).


In [ ]:
m, n = X_train.shape
print(f"Number of training examples (m): {m}")
print(f"Number of features (n): {n}")
print(f"y_train shape: {y_train.shape}")


#### Visualize the data

With two features we can no longer draw a single 2-D scatter of profit vs. everything, but we
can still look at each feature's relationship to profit separately, and check how correlated
the two features are with each other (an economist would call this checking for
**multicollinearity**).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(X_train[:, 0], y_train, marker='x', c='r')
axes[0].set_xlabel('Population (10,000s)')
axes[0].set_ylabel('Profit ($10,000s)')
axes[0].set_title('Profit vs. Population')

axes[1].scatter(X_train[:, 1], y_train, marker='x', c='g')
axes[1].set_xlabel('Median household income ($1,000s)')
axes[1].set_ylabel('Profit ($10,000s)')
axes[1].set_title('Profit vs. Income')

axes[2].scatter(X_train[:, 0], X_train[:, 1], marker='o', c='b', alpha=0.6)
axes[2].set_xlabel('Population (10,000s)')
axes[2].set_ylabel('Median household income ($1,000s)')
axes[2].set_title('Population vs. Income\n(checking for multicollinearity)')

plt.tight_layout()
plt.show()

print("Correlation between population and income:", np.corrcoef(X_train[:,0], X_train[:,1])[0,1].round(3))


<a name="4"></a>
## 4 - Refresher: Multiple Linear Regression

With $n$ features, the model becomes a **weighted sum plus an intercept**:

$$ f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = w_1 x_1^{(i)} + w_2 x_2^{(i)} + \dots + w_n x_n^{(i)} + b
 = \mathbf{w}\cdot\mathbf{x}^{(i)} + b$$

where $\mathbf{w} = [w_1, w_2, \dots, w_n]$ is now a **vector** of coefficients (one per
feature) instead of a single scalar. In our case $n=2$:

$$ f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = w_1 \cdot \text{population}^{(i)} + w_2 \cdot \text{income}^{(i)} + b $$

The **cost function** has exactly the same form as before, just with the vectorized prediction:

$$J(\mathbf{w},b) = \frac{1}{2m} \sum\limits_{i=0}^{m-1} \left(f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}\right)^2$$

And the **gradients** generalize to one partial derivative per weight, plus one for $b$:

$$\frac{\partial J(\mathbf{w},b)}{\partial w_j} = \frac{1}{m}\sum\limits_{i=0}^{m-1}\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}\right)x_j^{(i)}, \qquad j = 1,\dots,n$$
$$\frac{\partial J(\mathbf{w},b)}{\partial b} = \frac{1}{m}\sum\limits_{i=0}^{m-1}\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}\right)$$

**Economic reading of $w_j$:** each coefficient is a **marginal effect** — the expected change
in profit from a one-unit increase in that feature, *holding all other features constant*
(the "ceteris paribus" condition economists always attach to a regression coefficient).

**Implementation note:** with more than one feature, looping in Python is slow and error-prone.
We'll implement `compute_cost` and `compute_gradient` **vectorized** with NumPy matrix
operations instead of `for` loops over features — this is the standard approach once $n>1$.


<a name="5"></a>
## 5 - Compute Cost

<a name="ex01"></a>
### Exercise 1: `compute_cost`

Complete `compute_cost` below using **vectorized** NumPy operations (no Python loop over
examples or features):

1. Compute predictions for *all* examples at once: $\mathbf{f} = X\mathbf{w} + b$
   (in NumPy: `X @ w + b`, which broadcasts `b` across all `m` rows).
2. Compute the vector of errors: $\mathbf{f} - \mathbf{y}$.
3. Return $\dfrac{1}{2m}\sum (\text{errors})^2$.

If you get stuck, expand the hint below.


In [ ]:
# UNQ_C1
# GRADED FUNCTION: compute_cost

def compute_cost(X, y, w, b):
    """
    Computes the cost function for multiple linear regression.

    Args:
        X (ndarray): Shape (m, n) matrix of examples with n features each
        y (ndarray): Shape (m,)   vector of target values (profit)
        w (ndarray): Shape (n,)   vector of model weights (one per feature)
        b (scalar):                model bias/intercept

    Returns:
        total_cost (float): The cost of using w, b as the parameters for
            linear regression to fit the data points in X and y.
    """
    m = X.shape[0]

    # You need to return this variable correctly
    total_cost = 0.

    ### START CODE HERE ###

    ### END CODE HERE ###

    return total_cost


<details>
  <summary><font size="3" color="darkgreen"><b>Click for hints</b></font></summary>

* `X` has shape `(m, n)` and `w` has shape `(n,)`, so `X @ w` has shape `(m,)` — one prediction
  per example, computed for every feature simultaneously. This replaces the inner "loop over
  features" from a plain scalar implementation.
* Adding a scalar `b` to an `(m,)` array broadcasts it to every element automatically.
* `np.sum(errors ** 2)` sums the squared errors over all `m` examples in one call.

```python
def compute_cost(X, y, w, b):
    m = X.shape[0]
    predictions = X @ w + b
    errors = predictions - y
    total_cost = (1 / (2 * m)) * np.sum(errors ** 2)
    return total_cost
```
</details>


Run the cell below to sanity-check your implementation. Because the dataset is generated
with a fixed seed, the expected value below is exact (not approximate).

In [ ]:
# Sanity check with w = [0, 0], b = 0
initial_w = np.zeros(n)
initial_b = 0.

cost = compute_cost(X_train, y_train, initial_w, initial_b)
print(f"Cost at initial w (zeros), b=0: {cost:.4f}")

# --- lightweight inline test (no external public_tests.py needed) ---
assert np.isclose(cost, 121.1484, atol=1e-3), "compute_cost looks incorrect — check your formula."
print("compute_cost: all tests passed!")


**Expected Output**:
<table>
  <tr><td><b>Cost at initial w (zeros), b=0</b></td><td>121.1484</td></tr>
</table>


<a name="6"></a>
## 6 - Compute Gradient

<a name="ex02"></a>
### Exercise 2: `compute_gradient`

Complete `compute_gradient` below, again **fully vectorized**:

1. Compute the error vector as before: $\mathbf{f} - \mathbf{y}$ (shape `(m,)`).
2. $\dfrac{\partial J}{\partial \mathbf{w}} = \dfrac{1}{m} X^T (\mathbf{f}-\mathbf{y})$ — this single
   matrix-vector product computes **all $n$ partial derivatives at once** (shape `(n,)`),
   replacing what would otherwise be a nested loop over examples *and* features.
3. $\dfrac{\partial J}{\partial b} = \dfrac{1}{m}\sum(\mathbf{f}-\mathbf{y})$ (a scalar).


In [ ]:
# UNQ_C2
# GRADED FUNCTION: compute_gradient

def compute_gradient(X, y, w, b):
    """
    Computes the gradient for multiple linear regression.

    Args:
        X (ndarray): Shape (m, n) matrix of examples with n features each
        y (ndarray): Shape (m,)   vector of target values (profit)
        w (ndarray): Shape (n,)   vector of model weights
        b (scalar):                model bias/intercept

    Returns:
        dj_dw (ndarray): Shape (n,) gradient of the cost w.r.t. the parameters w
        dj_db (scalar):             gradient of the cost w.r.t. the parameter b
    """
    m, n_features = X.shape

    # You need to return the following variables correctly
    dj_dw = np.zeros(n_features)
    dj_db = 0.

    ### START CODE HERE ###

    ### END CODE HERE ###

    return dj_dw, dj_db


<details>
  <summary><font size="3" color="darkgreen"><b>Click for hints</b></font></summary>

* `X.T` has shape `(n, m)`. Multiplying `X.T @ errors` (shape `(n,)`) by `1/m` gives you
  `dj_dw` directly — no loop needed. This is the vectorized version of
  "for each feature j, sum errors[i] * X[i, j] over all i".
* `np.sum(errors)` sums the same error vector over all examples to get `dj_db`.

```python
def compute_gradient(X, y, w, b):
    m = X.shape[0]
    predictions = X @ w + b
    errors = predictions - y
    dj_dw = (1 / m) * (X.T @ errors)
    dj_db = (1 / m) * np.sum(errors)
    return dj_dw, dj_db
```
</details>


In [ ]:
# Sanity check with w = [0, 0], b = 0
tmp_dj_dw, tmp_dj_db = compute_gradient(X_train, y_train, initial_w, initial_b)
print("Gradient at initial w (zeros), b=0:")
print("  dj_dw =", tmp_dj_dw)
print("  dj_db =", tmp_dj_db)

# --- lightweight inline test ---
assert np.allclose(tmp_dj_dw, np.array([-233.2686, -975.0417]), atol=1e-3), "dj_dw looks incorrect."
assert np.isclose(tmp_dj_db, -13.3339, atol=1e-3), "dj_db looks incorrect."
print("compute_gradient: all tests passed!")


**Expected Output**:
<table>
  <tr><td><b>dj_dw</b></td><td>[-233.2686, -975.0417]</td></tr>
  <tr><td><b>dj_db</b></td><td>-13.3339</td></tr>
</table>

Notice how much larger the income gradient is than the population gradient, purely because
income is measured on a much bigger numeric scale (20-120) than population (2-25) — **not**
because income matters more economically. This is exactly the problem feature scaling fixes
next, and it's also why comparing *raw* coefficients across features is misleading without it.


<a name="7"></a>
## 7 - Feature Scaling

<a name="ex03"></a>
### Exercise 3: `zscore_normalize_features`

Population (range ~2-25) and income (range ~20-120) live on very different scales. Left
unscaled, gradient descent takes tiny, inefficient steps in the income direction and can
require a much smaller learning rate (or many more iterations) to converge. This is a genuinely
**new problem** that didn't exist in the one-feature lab, and it's one of the most important
practical differences between simple and multiple regression.

The fix is **z-score normalization**: for each feature $j$, subtract its mean and divide by its
standard deviation:

$$x^{(i)}_{j,\text{norm}} = \frac{x^{(i)}_j - \mu_j}{\sigma_j}$$

where $\mu_j$ is the mean and $\sigma_j$ is the standard deviation of feature $j$ **computed
across all training examples**. After normalization, every feature has mean 0 and standard
deviation 1, so gradient descent treats them fairly.

Complete `zscore_normalize_features` below:
1. Compute `mu`: the mean of each column of `X` (shape `(n,)`) — use `np.mean(X, axis=0)`.
2. Compute `sigma`: the standard deviation of each column of `X` (shape `(n,)`) — use
   `np.std(X, axis=0)`.
3. Compute `X_norm = (X - mu) / sigma` (broadcasting handles the per-column subtraction/division).


In [ ]:
# UNQ_C3
# GRADED FUNCTION: zscore_normalize_features

def zscore_normalize_features(X):
    """
    Computes z-score normalized features per column.

    Args:
        X (ndarray): Shape (m, n) input data, m examples, n features

    Returns:
        X_norm (ndarray): Shape (m, n) z-score normalized X
        mu (ndarray):     Shape (n,)   mean of each feature
        sigma (ndarray):  Shape (n,)   standard deviation of each feature
    """
    mu = np.zeros(X.shape[1])
    sigma = np.ones(X.shape[1])
    X_norm = X.copy()

    ### START CODE HERE ###

    ### END CODE HERE ###

    return X_norm, mu, sigma


<details>
  <summary><font size="3" color="darkgreen"><b>Click for hints</b></font></summary>

* `axis=0` means "compute this statistic down each column" — i.e., one mean and one std per
  feature, not one overall mean for the whole matrix.
* `X - mu` broadcasts a `(n,)` array against an `(m, n)` matrix by aligning columns, so each
  row has its own feature's mean subtracted correctly.

```python
def zscore_normalize_features(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma
```
</details>


In [ ]:
X_norm, mu, sigma = zscore_normalize_features(X_train)
print("mu (feature means):", mu)
print("sigma (feature std devs):", sigma)
print("First 5 rows of X_norm:\n", X_norm[:5])

# --- lightweight inline test ---
assert np.allclose(mu, np.array([13.8006, 70.3155]), atol=1e-3), "mu looks incorrect."
assert np.allclose(sigma, np.array([6.6019, 27.0636]), atol=1e-3), "sigma looks incorrect."
assert np.allclose(np.mean(X_norm, axis=0), 0, atol=1e-8), "X_norm should have ~0 mean per column."
assert np.allclose(np.std(X_norm, axis=0), 1, atol=1e-8), "X_norm should have ~1 std per column."
print("zscore_normalize_features: all tests passed!")


**Expected Output**:
<table>
  <tr><td><b>mu</b></td><td>[13.8006, 70.3155]</td></tr>
  <tr><td><b>sigma</b></td><td>[6.6019, 27.0636]</td></tr>
</table>


<a name="8"></a>
## 8 - Run Gradient Descent

With `compute_cost`, `compute_gradient`, and `zscore_normalize_features` complete, the training
loop itself is **identical in structure** to the one-variable lab — this is the payoff of
vectorizing: the same `gradient_descent` driver works whether `w` has 1 element or 1,000.
Nothing below needs to change as you add more features to a real project.


In [ ]:
def gradient_descent(X, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters):
    """
    Performs batch gradient descent to learn w, b. Updates w, b by taking
    num_iters gradient steps with learning rate alpha.
    """
    J_history = []
    w = copy.deepcopy(w_in)
    b = b_in

    for i in range(num_iters):
        dj_dw, dj_db = gradient_function(X, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db

        if i < 100000:
            J_history.append(cost_function(X, y, w, b))

        if i % math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:4}: Cost {J_history[-1]:8.4f}")

    return w, b, J_history

# Normalize features before training (see Exercise 3 discussion above)
X_norm, mu, sigma = zscore_normalize_features(X_train)

initial_w = np.zeros(n)
initial_b = 0.
alpha = 0.1
iterations = 1000

w_final, b_final, J_history = gradient_descent(
    X_norm, y_train, initial_w, initial_b,
    compute_cost, compute_gradient, alpha, iterations
)

print("\nw found by gradient descent (normalized-feature space):", w_final)
print("b found by gradient descent (normalized-feature space):", b_final)


**Expected Output** (normalized-feature space, `alpha=0.1`, `1000` iterations):
<table>
  <tr><td><b>w_final</b></td><td>[7.4361, 1.2386]</td></tr>
  <tr><td><b>b_final</b></td><td>13.3339</td></tr>
  <tr><td><b>final cost</b></td><td>≈ 3.66</td></tr>
</table>

Note these `w_final` values are in **normalized units** ("per one standard deviation of
population/income"), not dollars per person or dollars per dollar of income — we convert back
to interpretable raw units in Section 10.


#### Plot the cost history

A quick, essential diagnostic: cost should decrease monotonically and flatten out. If it's
oscillating or increasing, `alpha` is too large; if it's decreasing extremely slowly after
1000 iterations, `alpha` may be too small or you may need more iterations.


In [ ]:
plt.plot(J_history)
plt.title("Cost vs. iteration (gradient descent)")
plt.xlabel("Iteration")
plt.ylabel("Cost J(w,b)")
plt.show()


<a name="9"></a>
## 9 - Sanity Check: Closed-Form (OLS) Solution

Because ordinary least squares has a well-known **closed-form solution**
$\hat{\boldsymbol{\theta}} = (X_b^TX_b)^{-1}X_b^T\mathbf{y}$ (where $X_b$ is $X$ with a column
of 1's prepended for the intercept), we can check our gradient-descent answer against it
directly — this is a check you can *always* run for plain linear regression (it doesn't
generalize to most other models, which is exactly why gradient descent is worth learning).


In [ ]:
# Build design matrix with intercept column, using RAW (unnormalized) features here
X_design = np.column_stack([np.ones(m), X_train])
theta_closed_form = np.linalg.lstsq(X_design, y_train, rcond=None)[0]

b_closed, w1_closed, w2_closed = theta_closed_form
print("Closed-form solution (raw feature units):")
print(f"  b  = {b_closed:.4f}")
print(f"  w1 (population) = {w1_closed:.4f}")
print(f"  w2 (income)     = {w2_closed:.4f}")


<a name="10"></a>
## 10 - Economic Interpretation of Coefficients

`w_final`/`b_final` from gradient descent are in **normalized units** because we trained on
`X_norm`. To interpret them economically (dollars of profit per additional person, per
additional dollar of income), we need to convert back to the original, raw-feature scale:

$$ w_{j,\text{raw}} = \frac{w_{j,\text{norm}}}{\sigma_j}, \qquad
   b_{\text{raw}} = b_{\text{norm}} - \sum_j \frac{w_{j,\text{norm}} \cdot \mu_j}{\sigma_j} $$

This should match the closed-form solution from Section 9 almost exactly — a good end-to-end
check that both the gradient-descent implementation *and* the un-scaling math are correct.


In [ ]:
w_raw = w_final / sigma
b_raw = b_final - np.sum((w_final * mu) / sigma)

print("Gradient descent, converted to raw units:")
print(f"  w1 (population) = {w_raw[0]:.4f}")
print(f"  w2 (income)     = {w_raw[1]:.4f}")
print(f"  b               = {b_raw:.4f}")

print("\nCompare to closed-form (Section 9):")
print(f"  w1 = {w1_closed:.4f}, w2 = {w2_closed:.4f}, b = {b_closed:.4f}")


#### Reading the coefficients like an economist

- **$w_1 \approx 1.13$ (population):** holding median income fixed, each additional 10,000
  people in a city is associated with about **\$11,300 more** in average monthly restaurant
  profit. This is the **marginal effect of market size**.
- **$w_2 \approx 0.046$ (income):** holding population fixed, each additional \$1,000 of
  median household income is associated with about **\$460 more** in average monthly profit —
  the **marginal effect of purchasing power**.
- **$b \approx -5.43$ (intercept):** the model's fitted profit for a hypothetical city with
  *zero* population and *zero* income. It's a mathematical artifact of the linear fit, not an
  economically meaningful quantity here (no real city has zero population), which is a common
  caveat when interpreting intercepts in applied econometrics.
- **Recovered true parameters:** recall the data was simulated with `true_w = [1.15, 0.045]`
  and `true_b = -5.2` — the model recovered these closely, which is exactly what we'd hope for
  given random noise was added on top of a genuinely linear relationship.

A natural next question for an economist: which variable matters "more"? Comparing raw
coefficients directly is misleading (they're in different units — people vs. dollars). Two
better approaches: compare the **normalized coefficients** (`w_final`, which are already on a
common "per standard deviation" scale), or compute an **elasticity** (percentage-change
interpretation) below.


#### Optional: income elasticity of profit

Economists often prefer **elasticities** — the percent change in $y$ for a 1% change in $x_j$,
evaluated at the mean — because they're unit-free and easy to compare across variables:

$$ \text{Elasticity}_j = w_{j,\text{raw}} \cdot \frac{\bar{x}_j}{\bar{y}} $$


In [ ]:
x_bar = np.mean(X_train, axis=0)
y_bar = np.mean(y_train)

elasticity_population = w_raw[0] * (x_bar[0] / y_bar)
elasticity_income = w_raw[1] * (x_bar[1] / y_bar)

print(f"Elasticity of profit w.r.t. population: {elasticity_population:.3f}")
print(f"Elasticity of profit w.r.t. income:     {elasticity_income:.3f}")
print("\n(A 1% increase in population is associated with roughly a "
      f"{elasticity_population:.2f}% increase in profit, holding income fixed, and similarly for income.)")


<a name="11"></a>
## 11 - Model Fit Diagnostics (R²)

$R^2$ (the coefficient of determination) reports the fraction of the variance in profit that
the model explains:

$$ R^2 = 1 - \frac{\sum_i (y^{(i)} - f_{w,b}(x^{(i)}))^2}{\sum_i (y^{(i)} - \bar{y})^2} $$

An $R^2$ near 1 means the model explains most of the variation; near 0 means it explains
almost none. We'll also plot **residuals** (actual − predicted) against fitted values — in a
well-specified linear model, residuals should look like patternless noise scattered around 0,
with no funnel shape (heteroskedasticity) or curve (missing nonlinearity).


In [ ]:
predictions = X_norm @ w_final + b_final
residuals = y_train - predictions

ss_res = np.sum(residuals ** 2)
ss_tot = np.sum((y_train - np.mean(y_train)) ** 2)
r_squared = 1 - ss_res / ss_tot
print(f"R-squared: {r_squared:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_train, predictions, alpha=0.6)
lims = [min(y_train.min(), predictions.min()), max(y_train.max(), predictions.max())]
axes[0].plot(lims, lims, 'r--')
axes[0].set_xlabel("Actual profit")
axes[0].set_ylabel("Predicted profit")
axes[0].set_title(f"Actual vs. Predicted (R² = {r_squared:.3f})")

axes[1].scatter(predictions, residuals, alpha=0.6)
axes[1].axhline(0, color='r', linestyle='--')
axes[1].set_xlabel("Predicted profit")
axes[1].set_ylabel("Residual (actual - predicted)")
axes[1].set_title("Residual plot")

plt.tight_layout()
plt.show()


<a name="12"></a>
## 12 - Predicting for New Cities

To predict for a new candidate city, apply the **same** `mu`/`sigma` learned from the training
data (never recompute normalization stats on new data — that would leak information and give
inconsistent scaling), then apply the trained `w_final`, `b_final`.


In [ ]:
def predict_profit(population_10k, income_1k, mu, sigma, w, b):
    """
    Predicts monthly profit ($10,000s) for a city with given population (10,000s)
    and median household income ($1,000s), using a fitted normalized-feature model.
    """
    x_query = np.array([population_10k, income_1k])
    x_query_norm = (x_query - mu) / sigma
    return x_query_norm @ w + b

candidates = [
    (15.0, 60.0),   # 150,000 people, $60k median income
    (5.0, 100.0),   # 50,000 people, $100k median income (small, wealthy)
    (22.0, 30.0),   # 220,000 people, $30k median income (large, lower income)
]

for pop, inc in candidates:
    pred = predict_profit(pop, inc, mu, sigma, w_final, b_final)
    print(f"Population={pop*10000:>9,.0f}, Median income=${inc*1000:>9,.0f}  "
          f"-> Predicted monthly profit: ${pred*10000:>10,.2f}")


<a name="13"></a>
## 13 - Extension Questions

Try these to go deeper (no solutions provided — they're meant to mirror open-ended analysis you'd
do on a real project):

1. **Interaction effects:** add a feature `x3 = population * income`. Does it change the fit or
   the story ("wealthy AND populous cities are disproportionately profitable")?
2. **Diminishing returns:** try adding `population**2` as a feature. Is there evidence that
   profit grows sub-linearly with population once cities get very large?
3. **Regularization:** add an $L_2$ penalty $\frac{\lambda}{2m}\sum w_j^2$ to `compute_cost`
   and its gradient. How does increasing $\lambda$ shrink the coefficients toward zero, and at
   what point does $R^2$ start to suffer?
4. **Omitted-variable bias:** re-simulate the data with a third, *unobserved* driver of profit
   that's correlated with income (e.g., local rent), then fit the 2-feature model without it.
   How does the income coefficient's bias change?
5. **Out-of-sample validation:** split the 100 cities into train/test sets. Does the model's
   $R^2$ hold up on cities it didn't see during training?


**You've now extended one-variable linear regression to a full multiple linear regression /
econometric workflow: vectorized cost and gradient functions, feature scaling, a closed-form
cross-check, coefficient interpretation (marginal effects and elasticities), and residual
diagnostics. This same pattern — scale features, fit, un-scale coefficients, interpret, check
residuals — is the backbone of applied regression analysis in economics, and it's exactly what
the reusable template notebook packages up for your next dataset.**
